In [2]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.widgets import Slider
from scipy import signal

# If in Jupyter, this backend is usually needed for a working Slider
try:
    get_ipython().run_line_magic("matplotlib", "widget")
except Exception:
    pass



# ============================================================
# A) Open Ephys continuous.dat loader
# ============================================================

def _find_session_root_from_dat(dat_path: Path, max_up=12):
    """
    Walk upward from continuous.dat until we find a folder containing structure.oebin.
    That folder is the Open Ephys session root (the one containing Record Node ...).
    """
    p = Path(dat_path).resolve()
    for _ in range(max_up):
        if (p / "structure.oebin").is_file():
            return p
        if p.parent == p:
            break
        p = p.parent
    return None


def load_open_ephys_continuous_dat(dat_path: str | Path):
    """
    Load Open Ephys continuous.dat as (n_samples, n_channels) int16 array-like.

    First tries open-ephys-python-tools Session, then falls back to memmap using structure.oebin.
    """
    dat_path = Path(dat_path).resolve()
    if not dat_path.is_file():
        raise FileNotFoundError(f"continuous.dat not found: {dat_path}")

    stream_dir = dat_path.parent  # .../continuous/<stream>/
    recording_dir = dat_path.parents[2]  # .../experiment1/recordingX
    session_root = _find_session_root_from_dat(dat_path)
    if session_root is None:
        raise OSError(f"Could not find structure.oebin above: {dat_path}")

    # ---- Try open-ephys-python-tools ----
    try:
        from open_ephys.analysis import Session

        sess = Session(session_root)

        # Find the matching recording by folder name (e.g. "recording2")
        rec_name = recording_dir.name
        rec = None
        for r in sess.recordings:
            if Path(r.directory).name == rec_name:
                rec = r
                break
        if rec is None:
            if len(sess.recordings) == 1:
                rec = sess.recordings[0]
            else:
                raise FileNotFoundError(
                    f"Recording '{rec_name}' not found in session root '{session_root}'. "
                    f"Available: {[Path(r.directory).name for r in sess.recordings]}"
                )

        # Find matching continuous stream by folder name (e.g. "Acquisition_Board-100.Rhythm Data")
        stream_name = stream_dir.name
        cont = None
        for s in rec.continuous:
            if Path(s.directory).name == stream_name:
                cont = s
                break
        if cont is None:
            if len(rec.continuous) == 1:
                cont = rec.continuous[0]
            else:
                raise FileNotFoundError(
                    f"Continuous stream '{stream_name}' not found in recording '{rec_name}'. "
                    f"Available: {[Path(s.directory).name for s in rec.continuous]}"
                )

        data = cont.get_samples()  # usually int16
        return data

    except Exception as e:
        # ---- Fallback: parse structure.oebin and memmap continuous.dat ----
        oebin_path = session_root / "structure.oebin"
        if not oebin_path.is_file():
            raise OSError(
                "open_ephys Session failed and structure.oebin was not found.\n"
                f"Original error: {repr(e)}"
            )

        with open(oebin_path, "r") as f:
            meta = json.load(f)

        cont_entries = meta.get("continuous", [])
        if not cont_entries:
            raise OSError("structure.oebin has no 'continuous' entries.")

        # Match entry by folder name == stream folder
        stream_entry = None
        for entry in cont_entries:
            folder_name = entry.get("folder_name") or entry.get("folder") or ""
            if Path(folder_name).name == stream_dir.name:
                stream_entry = entry
                break

        if stream_entry is None and len(cont_entries) == 1:
            stream_entry = cont_entries[0]

        if stream_entry is None:
            raise OSError(
                f"Could not match stream '{stream_dir.name}' in structure.oebin."
            )

        num_ch = stream_entry.get("num_channels") or stream_entry.get("numChannels")
        if num_ch is None:
            raise OSError("structure.oebin missing num_channels for continuous stream.")

        file_bytes = dat_path.stat().st_size
        bytes_per_sample = 2  # int16
        denom = bytes_per_sample * int(num_ch)
        if file_bytes % denom != 0:
            raise OSError(
                "continuous.dat size not divisible by 2*num_channels; check metadata/stream."
            )

        n_samples = file_bytes // denom
        data = np.memmap(dat_path, dtype=np.int16, mode="r", shape=(n_samples, int(num_ch)))
        return data



# ============================================================
# B) Downsample continuous.dat -> cached .npy (NO overwrite)
# ============================================================

def downsample_continuous_dat_to_npy(
    continuous_dat_path,
    fs_raw=2000.0,
    fs_ds=1000.0,
    cache_dir=None,
    out_name=None,
    force_rebuild=False
):
    """
    Downsample continuous.dat to float32 .npy cache and return:
      lfp_ds (mmap), ratio, ds_path

    This does NOT modify/overwrite any original files.
    """
    continuous_dat_path = Path(continuous_dat_path).resolve()
    assert continuous_dat_path.is_file(), f"continuous.dat not found: {continuous_dat_path}"
    assert fs_raw % fs_ds == 0, "fs_raw must be integer multiple of fs_ds"
    ratio = int(fs_raw / fs_ds)

    if cache_dir is None:
        cache_dir = continuous_dat_path.parents[5] / "downsample_cache"
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)

    if out_name is None:
        rec_name = continuous_dat_path.parents[2].name  # recording2
        stream_name = continuous_dat_path.parent.name   # Acquisition_Board-100.Rhythm Data
        safe_stream = stream_name.replace(" ", "_").replace("/", "_")
        out_name = f"{rec_name}_{safe_stream}_DS{int(fs_ds)}Hz.npy"

    ds_path = cache_dir / out_name

    if ds_path.is_file() and not force_rebuild:
        lfp_ds = np.load(ds_path, mmap_mode="r")
        return lfp_ds, ratio, ds_path

    # Load raw
    lfp_raw = load_open_ephys_continuous_dat(continuous_dat_path)

    # Decimate
    lfp_ds_arr = signal.decimate(
        np.asarray(lfp_raw, dtype=np.float32),  # converts safely
        q=ratio,
        ftype="fir",
        axis=0,
        zero_phase=True
    ).astype(np.float32)

    np.save(ds_path, lfp_ds_arr)
    lfp_ds = np.load(ds_path, mmap_mode="r")
    return lfp_ds, ratio, ds_path



# ============================================================
# C) TTL extraction and conversion to indices
# ============================================================

def extract_ttl_pulses_raw(sample_numbers, states):
    """
    Return Nx2 array of [start, end] in RAW sample numbers.
    states: 1 rising, -1 falling
    """
    sample_numbers = np.asarray(sample_numbers)
    states = np.asarray(states)

    start_idx = np.where(states == 1)[0]
    end_idx   = np.where(states == -1)[0]
    n = min(len(start_idx), len(end_idx))
    if n == 0:
        return np.zeros((0, 2), dtype=np.int64)

    pulses = np.stack([sample_numbers[start_idx[:n]],
                       sample_numbers[end_idx[:n]]], axis=1)
    pulses = pulses[pulses[:, 1] > pulses[:, 0]]
    return pulses.astype(np.int64)


def load_ttl_as_lfp_indices(ttl_folder, cont_folder):
    """
    Load TTL pulses and convert to LFP RAW indices by subtracting cont0.
    Returns: pulses_idx_raw (Nx2) in raw LFP sample grid, relative to LFP start.
    """
    ttl_folder = Path(ttl_folder)
    cont_folder = Path(cont_folder)

    stim_samp = np.load(ttl_folder / "sample_numbers.npy")
    states    = np.load(ttl_folder / "states.npy")

    cont0 = int(np.load(cont_folder / "sample_numbers.npy", mmap_mode="r")[0])

    pulses_raw = extract_ttl_pulses_raw(stim_samp, states)
    pulses_idx = (pulses_raw - cont0).astype(np.int64)

    pulses_idx = pulses_idx[pulses_idx[:, 1] > pulses_idx[:, 0]]
    pulses_idx = pulses_idx[pulses_idx[:, 1] > 0]
    return pulses_idx



# ============================================================
# D) Position helpers
# ============================================================

def normalize_inside_roi(series):
    if series.dtype == bool:
        return series.values.astype(bool)
    return series.astype(str).str.upper().isin(["TRUE", "1", "T", "YES"]).values.astype(bool)


def segments_from_mask(mask, times_s, dt_est_s=0.0):
    mask = np.asarray(mask, dtype=bool)
    times_s = np.asarray(times_s, dtype=float)
    if mask.size == 0:
        return []

    edges = np.diff(mask.astype(np.int8), prepend=0, append=0)
    starts = np.where(edges == 1)[0]
    ends   = np.where(edges == -1)[0]

    segs = []
    for s, e in zip(starts, ends):
        if e <= s:
            continue
        t0 = float(times_s[s])
        t1 = float(times_s[e - 1] + dt_est_s)
        segs.append((t0, t1))
    return segs


def find_trial_start_pos(t_s, detected_mask, first_window_s=20.0, min_detected_run_s=3.0):
    """
    In the first first_window_s seconds (AFTER converting timestamps to seconds),
    find the first sample that begins a >=min_detected_run_s continuous detected run.
    """
    t_s = np.asarray(t_s, dtype=float)
    detected_mask = np.asarray(detected_mask, dtype=bool)
    if t_s.size < 2:
        return None

    dt = float(np.nanmedian(np.diff(t_s)))
    if not np.isfinite(dt) or dt <= 0:
        raise ValueError("Cannot estimate position dt; check timestamp conversion.")

    idx_limit = np.searchsorted(t_s, float(first_window_s), side="right")
    if idx_limit <= 1:
        return None

    m = detected_mask[:idx_limit]
    min_len = int(np.ceil(min_detected_run_s / dt))

    edges = np.diff(m.astype(np.int8), prepend=0, append=0)
    starts = np.where(edges == 1)[0]
    ends   = np.where(edges == -1)[0]

    for s, e in zip(starts, ends):
        if (e - s) >= min_len:
            return float(t_s[s])

    return None


def roi_onset_time(position_clean):
    roi = position_clean["inside_roi"].values.astype(bool)
    if roi.size == 0:
        return None
    onsets = np.where((roi[1:] == True) & (roi[:-1] == False))[0] + 1
    if onsets.size == 0 and roi[0]:
        return float(position_clean.loc[0, "t"])
    if onsets.size == 0:
        return None
    return float(position_clean.loc[onsets[0], "t"])


# ============================================================
# E) Alignment + slider sanity plot (uses DS LFP)
# ============================================================

def sanity_slider_alignment(
    continuous_dat_path,
    fs_raw=2000.0,
    fs_plot=1000.0,
    ds_cache_dir=None,
    force_rebuild_ds=False
,
    ttl_folder=None,
    cont_folder=None,
    pos_csv_path=None,
    channels=(9, 11, 13, 15),
    window_s=20.0
,
    pos_first_window_s=20.0,
    pos_min_detected_run_s=3.0
,
):
    """
    - continuous.dat -> DS cache (.npy)
    - TTL pulses -> raw indices -> DS indices (//ratio)
    - position rule: in first 10 s (seconds scale), first >=3 s detected run defines trial start
    - align first inside_roi onset to first TTL onset
    - plot: pre-trial dark grey, rest blue, fixed-height green dots, TTL spans, missing detection gaps
    """
    continuous_dat_path = Path(continuous_dat_path)
    pos_csv_path = Path(pos_csv_path) if pos_csv_path is not None else None
    ttl_folder = Path(ttl_folder)
    cont_folder = Path(cont_folder)

    assert continuous_dat_path.is_file(), f"continuous.dat not found: {continuous_dat_path}"
    assert (ttl_folder / "sample_numbers.npy").is_file(), f"TTL sample_numbers.npy not found: {ttl_folder}"
    assert (ttl_folder / "states.npy").is_file(), f"TTL states.npy not found: {ttl_folder}"
    assert (cont_folder / "sample_numbers.npy").is_file(), f"Continuous sample_numbers.npy not found: {cont_folder}"

    # ---- 1) Downsample continuous.dat -> cached .npy ----
    lfp_ds, ratio, ds_path = downsample_continuous_dat_to_npy(
        continuous_dat_path=continuous_dat_path,
        fs_raw=float(fs_raw),
        fs_ds=float(fs_plot),
        cache_dir=ds_cache_dir,
        force_rebuild=force_rebuild_ds
    )
    fs = float(fs_plot)
    n = lfp_ds.shape[0]
    t_lfp = np.arange(n) / fs

    print("Downsampled LFP loaded:", lfp_ds.shape, "fs =", fs, "| cache:", ds_path)
    print("Downsample ratio:", ratio)

    def _plot_lfp_only():
        fig, axes = plt.subplots(len(channels), 1, figsize=(12, 8), sharex=True)
        axes = np.atleast_1d(axes)
        plt.subplots_adjust(bottom=0.14, hspace=0.25)

        ax_sl = fig.add_axes([0.12, 0.05, 0.76, 0.035])
        max_start = max(0.0, float(t_lfp[-1]) - float(window_s))
        sld = Slider(ax_sl, "Start (s)", 0.0, max_start, valinit=0.0)
        fig._slider = sld

        COL_REST = "#1f77b4"  # blue

        def render(start_s):
            i0 = float(start_s)
            i1 = float(start_s + window_s)
            idx0 = int(i0 * fs)
            idx1 = int(min(n, i1 * fs))
            t = t_lfp[idx0:idx1]

            for ax in axes:
                ax.clear()

            for ax, ch in zip(axes, channels):
                seg = np.asarray(lfp_ds[idx0:idx1, ch], dtype=float)
                ax.plot(t, seg, lw=0.8, color=COL_REST)
                y_valid = seg[np.isfinite(seg)]
                if y_valid.size:
                    lo, hi = np.percentile(y_valid, [1, 99])
                    pad = 0.25 * (hi - lo + 1e-9)
                    ax.set_ylim(lo - pad, hi + pad)
                ax.set_ylabel(f"Ch {ch}")
                ax.grid(True, alpha=0.2)

            axes[-1].set_xlabel("Time (s)")
            axes[-1].set_xlim(i0, i1)
            fig.canvas.draw_idle()

        sld.on_changed(lambda v: render(sld.val))
        render(0.0)
        plt.show()

        return {
            "ds_path": str(ds_path),
            "ratio": ratio,
            "shift_s": None,
            "trial_start_pos_s": None,
            "trial_start_lfp_s": None,
            "ttl_spans": [],
            "inside_idx_lfp": np.array([], dtype=np.int64),
            "missing_segs_lfp": [],
        }

    # ---- 2) TTL pulses -> raw indices -> DS indices ----
    pulses_idx_raw = load_ttl_as_lfp_indices(ttl_folder, cont_folder)  # RAW grid
    if pulses_idx_raw.size == 0:
        print("No TTL pulses detected. Plotting LFP only (no TTL, no position alignment).")
        return _plot_lfp_only()

    pulses_ds = (pulses_idx_raw // ratio).astype(np.int64)

    pulses_ds[:, 0] = np.clip(pulses_ds[:, 0], 0, n - 1)
    pulses_ds[:, 1] = np.clip(pulses_ds[:, 1], 0, n - 1)
    pulses_ds = pulses_ds[pulses_ds[:, 1] > pulses_ds[:, 0]]
    if len(pulses_ds) == 0:
        print("No TTL pulses left after downsampling/clipping. Plotting LFP only.")
        return _plot_lfp_only()

    ttl_spans = [(float(s / fs), float(e / fs)) for s, e in pulses_ds]
    first_ttl_s = ttl_spans[0][0]

    if pos_csv_path is None or not pos_csv_path.is_file():
        raise ValueError("Position CSV is required when TTL pulses are present.")

    # ---- 3) Position load + time conversion to seconds ----
    pos = pd.read_csv(pos_csv_path)
    required_cols = {"frame", "timestamp", "smooth_trans_x", "smooth_trans_y", "inside_roi"}
    missing = required_cols - set(pos.columns)
    if missing:
        raise ValueError(f"Missing columns in position file: {missing}")

    pos = pos[["frame", "timestamp", "smooth_trans_x", "smooth_trans_y", "inside_roi"]].copy()
    pos["inside_roi"] = normalize_inside_roi(pos["inside_roi"])

    # Convert timestamp to seconds (auto-detect ms vs s)
    ts = pos["timestamp"].astype(float).values
    dt_ts = float(np.nanmedian(np.diff(ts))) if ts.size > 1 else np.nan
    if np.isfinite(dt_ts) and dt_ts > 1.0:
        t_pos = (ts - ts[0]) / 1000.0
    else:
        t_pos = ts - ts[0]
    pos["t"] = t_pos

    detected_mask_raw = (pos["smooth_trans_x"].values != -1)

    # ---- 4) Trial start rule (in first 10 s of *seconds* scale) ----
    trial_start_pos_s = find_trial_start_pos(
        t_s=pos["t"].values,
        detected_mask=detected_mask_raw,
        first_window_s=float(pos_first_window_s),
        min_detected_run_s=float(pos_min_detected_run_s)
    )
    if trial_start_pos_s is None:
        raise ValueError("Could not find a >=3s detected run within first window to define trial start.")

    # ---- 5) Clean position (drop undetected) for ROI onset ----
    pos_clean = pos[detected_mask_raw].reset_index(drop=True)
    if len(pos_clean) == 0:
        raise ValueError("No detected position samples after removing smooth_trans_x == -1.")

    first_inside_t = roi_onset_time(pos_clean)
    if first_inside_t is None:
        raise ValueError("inside_roi never turns TRUE in detected position data.")

    # ---- 6) Align ROI onset to TTL onset ----
    shift_s = float(first_ttl_s - first_inside_t)
    trial_start_lfp_s = float(trial_start_pos_s + shift_s)

    print("Position trial start (s):", trial_start_pos_s)
    print("Trial start in LFP time (s):", trial_start_lfp_s)
    print("shift_s:", shift_s)
    print("TTL pulses (DS):", len(ttl_spans))
    print("inside_roi TRUE count (frames):", int(pos_clean["inside_roi"].sum()))

    # ---- 7) Missing detection segments -> LFP time ----
    dt_pos = float(np.nanmedian(np.diff(pos["t"].values))) if len(pos) > 1 else 0.0
    miss_mask = ~detected_mask_raw
    missing_segs_pos = segments_from_mask(miss_mask, pos["t"].values, dt_est_s=dt_pos)
    missing_segs_lfp = [(s + shift_s, e + shift_s) for s, e in missing_segs_pos]

    # ---- 8) ALL inside_roi TRUE frames -> DS indices ----
    inside_times_lfp = (pos_clean.loc[pos_clean["inside_roi"], "t"].values + shift_s)
    inside_idx_lfp = np.floor(inside_times_lfp * fs).astype(np.int64)
    inside_idx_lfp = np.unique(inside_idx_lfp)
    inside_idx_lfp = inside_idx_lfp[(inside_idx_lfp >= 0) & (inside_idx_lfp < n)]
    print("inside_roi dots (DS indices):", inside_idx_lfp.size)

    # ---- 9) Front removed region ----
    front_removed_end = max(0.0, trial_start_lfp_s)
    front_removed_seg = (0.0, front_removed_end)

    # ---- helper: apply missing segments as NaNs in the displayed window ----
    def apply_missing_nan(seg, t0, fs):
        out = seg.copy()
        if out.size == 0:
            return out
        win_end = t0 + out.size / fs
        for s, e in missing_segs_lfp:
            if e <= t0 or s >= win_end:
                continue
            i0 = int(max(0, np.floor((s - t0) * fs)))
            i1 = int(min(out.size, np.ceil((e - t0) * fs)))
            if i1 > i0:
                out[i0:i1] = np.nan
        return out

    # ---- 10) Slider plot ----
    fig, axes = plt.subplots(len(channels) + 1, 1, figsize=(12, 9), sharex=True)
    plt.subplots_adjust(bottom=0.14, hspace=0.25)

    ax_sl = fig.add_axes([0.12, 0.05, 0.76, 0.035])
    max_start = max(0.0, float(t_lfp[-1]) - float(window_s))
    sld = Slider(ax_sl, "Start (s)", 0.0, max_start, valinit=0.0)
    fig._slider = sld  # prevent GC

    COL_REST = "#1f77b4"  # blue
    COL_PRE  = "#4d4d4d"  # dark grey
    COL_DOTS = "#00B050"  # green

    def shade_front(ax, i0, i1):
        s = max(front_removed_seg[0], i0)
        e = min(front_removed_seg[1], i1)
        if e > s:
            ax.axvspan(s, e, color="#b0b0b0", alpha=0.25, lw=0)

    def shade_ttl(ax, i0, i1, alpha=0.10):
        for s, e in ttl_spans:
            if e <= i0 or s >= i1:
                continue
            ax.axvspan(max(s, i0), min(e, i1), alpha=alpha, color="black", lw=0)

    def render(start_s):
        i0 = float(start_s)
        i1 = float(start_s + window_s)

        idx0 = int(i0 * fs)
        idx1 = int(min(n, i1 * fs))

        t = t_lfp[idx0:idx1]

        for ax in axes:
            ax.clear()

        inside_idx_win = inside_idx_lfp[(inside_idx_lfp >= idx0) & (inside_idx_lfp < idx1)]
        t_inside = t_lfp[inside_idx_win] if inside_idx_win.size else np.array([])

        # LFP panels
        for ax, ch in zip(axes[:-1], channels):
            seg = np.asarray(lfp_ds[idx0:idx1, ch], dtype=float)
            seg = apply_missing_nan(seg, t0=i0, fs=fs)

            pre_mask = (t < front_removed_end)
            seg_pre = seg.copy()
            seg_rest = seg.copy()
            seg_pre[~pre_mask] = np.nan
            seg_rest[pre_mask] = np.nan

            ax.plot(t, seg_pre,  lw=0.8, color=COL_PRE)
            ax.plot(t, seg_rest, lw=0.8, color=COL_REST)

            shade_front(ax, i0, i1)
            shade_ttl(ax, i0, i1)

            # fixed-height dots (same height)
            y_valid = seg[np.isfinite(seg)]
            if y_valid.size:
                lo, hi = np.percentile(y_valid, [1, 99])
                pad = 0.25 * (hi - lo + 1e-9)
                ax.set_ylim(lo - pad, hi + pad)
                y_dot = hi + 0.10 * (hi - lo + 1e-9)
            else:
                ax.set_ylim(-1, 1)
                y_dot = 0.8

            if t_inside.size:
                ax.scatter(
                    t_inside,
                    np.full_like(t_inside, y_dot, dtype=float),
                    s=45,
                    color=COL_DOTS,
                    alpha=0.95,
                    zorder=5
                )

            ax.set_ylabel(f"Ch {ch}")
            ax.grid(True, alpha=0.2)

        # TTL panel
        axes[-1].plot(t, np.zeros_like(t), lw=0.8, color="0.2")
        for s, e in ttl_spans:
            if e <= i0 or s >= i1:
                continue
            axes[-1].axvspan(max(s, i0), min(e, i1), color="0.2", alpha=0.35, lw=0)

        shade_front(axes[-1], i0, i1)
        axes[-1].set_ylabel("TTL")
        axes[-1].set_xlabel("Time (s)")
        axes[-1].set_ylim(-1, 1)
        axes[-1].grid(True, alpha=0.2)
        axes[-1].set_xlim(i0, i1)

        fig.suptitle(
            f"{continuous_dat_path.name} | {pos_csv_path.name}\n"
            f"DS cache: {Path(ds_path).name} | fs_raw={fs_raw:g}Hz -> fs_ds={fs:g}Hz | ratio={ratio}\n"
            f"shift_s={shift_s:.3f}s | trial_start_lfp_s={trial_start_lfp_s:.3f}s | TTL0={first_ttl_s:.3f}s",
            y=0.99, fontsize=10
        )

        fig.canvas.draw_idle()

    sld.on_changed(lambda v: render(sld.val))
    render(0.0)
    plt.show()

    return {
        "ds_path": str(ds_path),
        "ratio": ratio,
        "shift_s": shift_s,
        "trial_start_pos_s": trial_start_pos_s,
        "trial_start_lfp_s": trial_start_lfp_s,
        "ttl_spans": ttl_spans,
        "inside_idx_lfp": inside_idx_lfp,
        "missing_segs_lfp": missing_segs_lfp,
    }





In [ ]:
# ============================================================
# Example run (edit these paths)
# ============================================================

base_dir = Path("/Users/stella/Desktop/PhD/paulsen_lab/Analysis/kou_analysis/cheeseboard-data/derivatives/sub-016_id-AM/ses-08_date-20250801/")

rec = base_dir / "ephys/recording1"

continuous_dat = rec / "continuous/Acquisition_Board-100.Rhythm Data/continuous.dat"

# MUST be the same acquisition stream folder for cont0 alignment
ttl_folder  = rec / "events/Acquisition_Board-100.Rhythm Data/TTL"
cont_folder = rec / "continuous/Acquisition_Board-100.Rhythm Data"

pos_csv = base_dir / "behav/sub-016_ses-08_trial-01_data-position.csv"

out = sanity_slider_alignment(
    continuous_dat_path=continuous_dat,
    fs_raw=2000.0,
    fs_plot=1000.0,
    ds_cache_dir=base_dir / "downsample_cache",
    force_rebuild_ds=False
,
    ttl_folder=ttl_folder,
    cont_folder=cont_folder,
    pos_csv_path=pos_csv
,
    channels=(9, 11, 13, 15),
    window_s=20.0
,
    pos_first_window_s=20.0,
    pos_min_detected_run_s=3.0
,
)

print("DONE")
print("Downsample cache:", out["ds_path"])
print("ratio:", out["ratio"])
print("shift_s:", out["shift_s"])
print("trial_start_lfp_s:", out["trial_start_lfp_s"])
print("TTL pulses:", len(out["ttl_spans"]))
print("inside_roi dots:", len(out["inside_idx_lfp"]))

AssertionError: continuous.dat not found: /Users/stella/Desktop/PhD/paulsen_lab/Analysis/kou_analysis/cheeseboard-data/derivativessub-016_id-AM/ses-08_date-20250801/ephys/recording1/continuous/Acquisition_Board-100.Rhythm Data/continuous.dat